In [ ]:
pip install mlflow dagshub optuna xgboost imbalanced-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.

In [ ]:
from google.colab import userdata
import os


os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")

In [ ]:

import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/")

In [ ]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos for yt")

<Experiment: artifact_location='mlflow-artifacts:/213dc38cd2f74c94aea47e2200a269f4', creation_time=1784485031768, effective_trace_archival_retention=None, experiment_id='13', last_update_time=1784485031768, lifecycle_stage='active', name='Exp 5 - ML Algos for yt', tags={}, trace_location=None, workspace='default'>

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from lightgbm import LGBMClassifier
import mlflow
import mlflow.sklearn
import optuna
from imblearn.over_sampling import RandomOverSampler

In [ ]:
df = pd.read_csv("/content/yt_preprocessed_data.csv").dropna(subset=['Comment'])
df.shape

(17680, 3)

In [ ]:
df.drop(columns=[ "Unnamed: 0"], inplace=True)

In [ ]:
df.head()

,Comment,Sentiment
0,let not forget apple pay 2014 required brand n...,neutral
1,nz 50 retailer dont even contactless credit ca...,negative
2,forever acknowledge channel help lesson idea e...,positive
3,whenever go place doesnt take apple pay doesnt...,negative
4,apple pay convenient secure easy use used kore...,positive


In [ ]:
df['Sentiment'] = df['Sentiment'].map({
    "neutral": 0,
    "positive": 1,
    "negative": 2
})

X_train, X_test, y_train, y_test = train_test_split(
    df["Comment"],
    df["Sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["Sentiment"]
)

vectorizer = CountVectorizer(
    max_features=1000,
    ngram_range=(1,3)
)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

filter = RandomOverSampler()
X_train , y_train = filter.fit_resample(X_train,y_train)



In [ ]:
import numpy as np

In [ ]:
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

In [ ]:
def ml_log(model_name , model , X_train , y_train , X_test , y_test):
  with mlflow.start_run():
      #log model type
      mlflow.set_tag('mlflow.runName' , f"{model_name}")
      mlflow.set_tag("experiment_type", "algorithm_comparison")

      # Log algorithm name as a parameter
      mlflow.log_param("algo_name", model_name)

      model.fit(X_train, y_train)
      y_pred = model.predict(X_test)

      #log accuracy
      accuracy = accuracy_score(y_test,y_pred)
      mlflow.log_metric("accuracy",accuracy)

        # Log classification report
      classification_rep = classification_report(y_test, y_pred, output_dict=True)
      for label, metrics in classification_rep.items():
          if isinstance(metrics, dict):
              for metric, value in metrics.items():
                  mlflow.log_metric(f"{label}_{metric}", value)


        # Log the model
      mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path=f"{model_name}_model",
            serialization_format="pickle"
      )

In [ ]:
# Step 6: Optuna objective function for LightGBM
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    model = LGBMClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, random_state=42)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))

In [ ]:
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42)

    # Log the best model with MLflow, passing the algo_name as "LightGBM"
    ml_log(
      "LightGBM",
      best_model,
      X_train,
      y_train,
      X_test,
      y_test
    )

# Run the experiment for LightGBM
run_optuna_experiment()

[I 2026-07-20 10:04:57,014] A new study created in memory with name: no-name-a4b3faa9-f7ac-482e-b648-ca793222cb78


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.165839 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4023
[LightGBM] [Info] Number of data points in the train set: 26484, number of used features: 1000
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] N

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-20 10:05:00,881] Trial 0 finished with value: 0.6572398190045249 and parameters: {'n_estimators': 300, 'learning_rate': 0.03329685375519809, 'max_depth': 5}. Best is trial 0 with value: 0.6572398190045249.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.174864 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4023
[LightGBM] [Info] Number of data points in the train set: 26484, number of used features: 1000
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-20 10:05:01,714] Trial 1 finished with value: 0.5921945701357466 and parameters: {'n_estimators': 51, 'learning_rate': 0.0869961503120166, 'max_depth': 3}. Best is trial 0 with value: 0.6572398190045249.


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.167019 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4023
[LightGBM] [Info] Number of data points in the train set: 26484, number of used features: 1000
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-20 10:06:59,912] Trial 29 finished with value: 0.682975113122172 and parameters: {'n_estimators': 296, 'learning_rate': 0.031140809511811668, 'max_depth': 10}. Best is trial 11 with value: 0.7095588235294118.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.162995 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4023
[LightGBM] [Info] Number of data points in the train set: 26484, number of used features: 1000
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/07/20 10:07:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/20 10:07:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/13/runs/978fe8a42400400abf0c90cabcb145c4
🧪 View experiment at: https://dagshub.com/panchariyarohit486/youtube-sentiment-analysis.mlflow/#/experiments/13


In [ ]:
print(type(y_train))
print(y_train.shape)

from scipy.sparse import issparse
print(issparse(y_train))

<class 'pandas.core.series.Series'>
(26484,)
False


In [ ]:
print(type(X_train), X_train.dtype)
print(type(X_test), X_test.dtype)

print(type(y_train), y_train.dtype)
print(type(y_test), y_test.dtype)

<class 'scipy.sparse._csr.csr_matrix'> int64
<class 'scipy.sparse._csr.csr_matrix'> int64
<class 'pandas.core.series.Series'> int64
<class 'pandas.core.series.Series'> int64
